# ch03 — Gen2: Matrix Profile discord

stumpy로 주기 신호 속 파형 왜곡(discord)을 찾는다.
UCR 데이터가 있으면 (`tsad-forge download ucr`) 실데이터로 바꿔 실행해 보라.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
import stumpy

t = np.arange(3000)
x = np.sin(2 * np.pi * t / 60) + 0.05 * np.random.default_rng(0).normal(size=3000)
x[1500:1560] = np.sin(2 * np.pi * t[1500:1560] / 17)  # 주기 붕괴 discord

m = 60
mp = stumpy.stump(x, m=m)[:, 0].astype(float)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
a1.plot(x, lw=0.5); a1.axvspan(1500, 1560, alpha=0.3, color="red"); a1.set_title("series")
a2.plot(mp, lw=0.7); a2.set_title(f"matrix profile (m={m}) — 최대값 위치가 discord")
print("discord at:", int(np.argmax(mp)))

In [ ]:
# IForest는 시드에 따라 결과가 다르다 — 그래서 시드 3개 평가 (CLAUDE.md §4)
from tsad_forge.evaluation.metrics import compute_metrics
from tsad_forge.models.registry import get_model

ds = generate_synthetic(seed=0)
for seed in range(3):
    s = get_model("iforest", seed=seed).fit(ds.train).score(ds.test)
    print(f"seed={seed}: VUS-PR={compute_metrics(s, ds.labels)['vus_pr']:.3f}")